# 08 — Liên hệ Thiên lệch Đầu ra với Hình học Embedding & Kiểm tra Ổn định qua 2 Seed

Notebook này kết nối hai góc nhìn của **SV1 (Popularity Bias ở đầu ra)** và **SV2 (Hình học không gian Embedding)** trên cùng bảng phân tích cấp độ item (`results/item_level/*.parquet`), đồng thời kiểm tra tính ổn định của mọi kết luận qua **2 random seed (`2024` và `2025`)**.

## Nội dung phân tích chính (Mục 12 & Mục 15 tài liệu đề tài)

1. **Ba tương quan cốt lõi (Spearman $\rho$, p-value, 95% Bootstrap CI):**
   - `train_freq ↔ embedding_norm`: Item phổ biến trong train có vector dài hơn không?
   - `train_freq ↔ rec_count`: Item phổ biến trong train có được gợi ý nhiều hơn trong Top-20 không?
   - `embedding_norm ↔ rec_count`: Item có norm lớn hơn có được gợi ý nhiều hơn không?
2. **Kiểm soát biến gây nhiễu `train_frequency`:**
   - Tính tương quan `embedding_norm ↔ rec_count` bên trong từng bin tần suất (`rare`, `medium`, `popular`).
   - Hồi quy đa biến $\log(1 + \text{rec\_count}_i) = \beta_0 + \beta_1 \log(1 + \text{train\_freq}_i) + \beta_2 \lVert e_i \rVert_2$.
3. **Kiểm tra độ ổn định qua 2 seed (`seed_stability.csv`) và Bảng tổng hợp cuối cùng (`final_model_comparison.csv`).**

## 1. Nạp dữ liệu từ `results/item_level/` và `results/tables/`

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án BTL-K03.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import FIGURES_DIR, ITEM_LEVEL_DIR, STAT_TESTS_DIR, TABLES_DIR, ensure_output_dirs
from btl_k03.embedding_analysis import analyze_three_core_relationships
from btl_k03.utils import load_experiment_config

ensure_output_dirs()

COMMON_CFG = load_experiment_config("gru4rec")
MODELS = ["gru4rec", "narm", "sasrec"]
SEEDS = COMMON_CFG["seeds"]

item_frames = []
for m in MODELS:
    for s in SEEDS:
        path = ITEM_LEVEL_DIR / f"{m}_seed{s}.parquet"
        if not path.is_file():
            raise FileNotFoundError(f"Thiếu {path}. Hãy chạy các notebook 03, 04, 05 trước!")
        item_frames.append(pd.read_parquet(path))

all_item_df = pd.concat(item_frames, ignore_index=True)
accuracy_df = pd.read_csv(TABLES_DIR / "model_accuracy.csv")
bias_df = pd.read_csv(TABLES_DIR / "popularity_bias.csv")
cosine_df = pd.read_csv(TABLES_DIR / "cosine_category.csv")

print(f"Đã nạp thành công {len(all_item_df):,} dòng dữ liệu item-level từ 6 lần chạy (3 models x 2 seeds).")

## 2. Phân tích 3 tương quan cốt lõi, tương quan trong từng Frequency Bin và Hồi quy đa biến

In [ ]:
core_corr_frames = []
bin_corr_frames = []
reg_rows = []

for (m, s), group_df in all_item_df.groupby(["model", "seed"], sort=False):
    c_df, b_df, r_dict = analyze_three_core_relationships(group_df, n_bootstrap=500, seed=int(s))
    core_corr_frames.append(c_df)
    bin_corr_frames.append(b_df)
    reg_rows.append(r_dict)

core_corr_df = pd.concat(core_corr_frames, ignore_index=True)
bin_corr_df = pd.concat(bin_corr_frames, ignore_index=True)
reg_df = pd.DataFrame(reg_rows)

core_corr_df.to_csv(TABLES_DIR / "bias_geometry_relationship.csv", index=False)
bin_corr_df.to_csv(STAT_TESTS_DIR / "within_bin_correlations.csv", index=False)
reg_df.to_csv(STAT_TESTS_DIR / "ols_regression_results.csv", index=False)

print("=== BẢNG 1: BA TƯƠNG QUAN SPEARMAN CỐT LÕI (FREQ ↔ NORM, FREQ ↔ REC, NORM ↔ REC) ===")
display(core_corr_df.loc[core_corr_df["scope"] == "warm_only"])

print("\n=== BẢNG 2: TƯƠNG QUAN NORM ↔ REC_COUNT TRONG TỪNG NHÓM TẦN SUẤT (KIỂM SOÁT FREQUENCY) ===")
display(bin_corr_df)

print("\n=== BẢNG 3: HỒI QUY ĐA BIẾN log(1 + rec_count) = β0 + β1*log(1 + train_freq) + β2*norm ===")
display(reg_df)

## 3. Bảng Tổng hợp So sánh Cuối cùng & Kiểm tra Độ ổn định qua 2 Seed

In [ ]:
# Gom ba tương quan chính về dạng cột cho từng (model, seed)
warm_corrs = core_corr_df.loc[core_corr_df["scope"] == "warm_only"].copy()
warm_corrs["model"] = warm_corrs["model"].str.upper()
piv_corr = warm_corrs.pivot(index=["model", "seed"], columns="pair", values="rho").reset_index()
piv_corr = piv_corr.rename(columns={
    "train_freq <-> embedding_norm": "spearman_freq_norm",
    "train_freq <-> rec_count": "spearman_freq_rec",
    "embedding_norm <-> rec_count": "spearman_norm_rec",
})

# Lấy Cosine cùng category và khác category
piv_cos = cosine_df.pivot(index=["model", "seed"], columns="pair_group", values="mean_cosine").reset_index()

full_run_table = (
    accuracy_df[["model", "seed", "recall@20", "mrr@20", "ndcg@20", "last_item_recall@20", "full_test_effective_recall@20"]]
    .merge(
        bias_df[["model", "seed", "coverage_warm@20", "gini_warm@20", "arp@20", "popular_group_exposure_share_pct", "rare_group_exposure_share_pct"]],
        on=["model", "seed"],
    )
    .merge(piv_corr, on=["model", "seed"])
    .merge(
        piv_cos[["model", "seed", "warm_same_category", "warm_diff_category", "cold_warm_same_category"]],
        on=["model", "seed"],
    )
)

full_run_table.to_csv(TABLES_DIR / "seed_stability.csv", index=False)
print("=== BẢNG 4: BẢNG CHI TIẾT TỪNG SEED (KIỂM TRA ĐỘ ỔN ĐỊNH QUA 2 SEED 2024 & 2025) ===")
display(full_run_table)

final_comparison = (
    full_run_table.drop(columns=["seed"])
    .groupby("model", sort=False)
    .agg(["mean", "std"])
)
final_comparison.columns = [f"{col}_{stat}" for col, stat in final_comparison.columns]
final_comparison = final_comparison.reset_index()
final_comparison.to_csv(TABLES_DIR / "final_model_comparison.csv", index=False)

print("\n=== BẢNG 5: BẢNG TỔNG HỢP CUỐI CÙNG (MEAN ± STD QUA 2 SEED) TRẢ LỜI 3 CÂU HỎI ĐỀ TÀI ===")
display(final_comparison)

## 4. Biểu đồ Liên hệ giữa Hình học Embedding và Tần suất được Gợi ý

In [ ]:
first_seed = SEEDS[0]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
group_colors = {"popular": "#d95f02", "medium": "#7570b3", "rare": "#1b9e77"}

for idx, m in enumerate(MODELS):
    sub = all_item_df.loc[
        (all_item_df["model"] == m) & (all_item_df["seed"] == first_seed) & (all_item_df["is_trainable_warm"])
    ].copy()
    sub["log10_rec_count"] = np.log10(1.0 + sub["rec_count"])
    rho_nr = full_run_table.loc[
        (full_run_table["model"] == m.upper()) & (full_run_table["seed"] == first_seed),
        "spearman_norm_rec",
    ].iloc[0]

    sns.scatterplot(
        data=sub.sample(n=min(6000, len(sub)), random_state=first_seed),
        x="embedding_norm",
        y="log10_rec_count",
        hue="popularity_group",
        hue_order=["popular", "medium", "rare"],
        palette=group_colors,
        alpha=0.45,
        s=18,
        ax=axes[idx],
    )
    axes[idx].set_title(f"{m.upper()} — Norm vs log10(1 + rec_count)\nSpearman ρ = {rho_nr:.4f}")
    axes[idx].set_xlabel("L2 Norm ||e_i||_2")
    axes[idx].set_ylabel("log10(1 + Recommendation Count)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "norm_recommendation_relationship.png", dpi=300)
plt.show()

# Hình tổng hợp so sánh 3 mô hình trên 4 chỉ số chính: Recall@20, Coverage@20, Gini@20, Spearman(freq, norm)
fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))
metrics_to_plot = [
    ("recall@20", "Recall@20 (Warm Test) ↑", "#2b5c8f"),
    ("coverage_warm@20", "Catalog Coverage@20 ↑", "#1b9e77"),
    ("gini_warm@20", "Recommendation Gini@20 ↓", "#d95f02"),
    ("spearman_freq_norm", "Spearman ρ (Freq ↔ Norm)", "#7570b3"),
]

for idx, (col, title, color) in enumerate(metrics_to_plot):
    sns.barplot(data=full_run_table, x="model", y=col, hue="seed", palette="Set2", ax=axes[idx])
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt="%.3f", padding=2, fontsize=9)
    axes[idx].set_title(title)
    axes[idx].set_xlabel("Mô hình")
    axes[idx].set_ylabel(col)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "final_model_comparison.png", dpi=300)
plt.show()